# Intraday recovery direct SOXL/SOXX/cash allocation

假设：日频模型的主要问题不是缺少趋势 alpha，而是回撤期用全天 cash 防守，导致恢复过慢。前四根 15 分钟 bar 只在 10:30 后使用，模型直接输出合法三资产仓位。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.workflow.intraday_recovery_direct_allocation import run_intraday_recovery_direct_allocation_card

summary = run_intraday_recovery_direct_allocation_card(root=ROOT)
summary['decision']

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[59014:MainThread](2026-07-24 16:15:59,633) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[59014:MainThread](2026-07-24 16:15:59,991) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[59014:MainThread](2026-07-24 16:15:59,991) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[59014:MainThread](2026-07-24 16:16:18,068) INFO - qlib.workflow - [exp.py:258] - Experiment 903361124573160811 starts running ...


[59014:MainThread](2026-07-24 16:16:18,119) INFO - qlib.workflow - [recorder.py:345] - Recorder e5c1bf5e491947938535fca3ec490949 starts running under Experiment 903361124573160811 ...


[59014:MainThread](2026-07-24 16:16:18,194) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | waiting `async_log` Done


'promoted_research_candidate_intraday_recovery_direct_allocation'

In [2]:
CARD = ROOT / 'research' / 'intraday_recovery_direct_allocation_v1'
display(pd.read_csv(CARD / 'risk_analysis.csv'))
display(pd.read_csv(CARD / 'annual_metrics.csv'))
display(pd.read_csv(CARD / 'cost_sensitivity.csv'))

,variant,segment,cost_bps,terminal_wealth,return_pct,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,avg_cash_weight_pct
0,daily_risk_controller,outer_diagnostic,10.0,6.405320,540.531969,-27.445163,-9.382462,-5.202018,-11.301752,37.759811
1,selected_direct_intraday_recovery,outer_diagnostic,10.0,7.296689,629.668873,-28.105120,-9.382462,-5.253781,-11.301752,35.828892
2,selected_direct_intraday_recovery_fixed_decisions,outer_diagnostic,25.0,6.574358,557.435830,-29.088729,-9.382462,-5.262269,-11.371385,35.828892


,year,terminal_wealth,return_pct,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,terminal_wealth_ratio_vs_soxx,avg_cash_weight_pct,recovery_action_count
0,2023,1.417019,41.701863,-21.654721,-5.401278,-4.267412,-8.145401,0.903588,33.607852,3
1,2024,1.393221,39.322146,-18.027346,-7.390744,-4.679965,-10.092335,1.153523,41.863090,8
2,2025,1.619375,61.937479,-22.933970,-7.185349,-4.954945,-10.535412,1.094333,36.770165,14
3,2026,2.282353,128.235256,-21.362152,-9.382462,-7.132735,-11.301752,1.186647,25.905950,5


,variant,segment,cost_bps,terminal_wealth,return_pct,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,avg_cash_weight_pct
0,selected_direct_intraday_recovery,outer_diagnostic,10.0,7.296689,629.668873,-28.105120,-9.382462,-5.253781,-11.301752,35.828892
1,selected_direct_intraday_recovery_fixed_decisions,outer_diagnostic,25.0,6.574358,557.435830,-29.088729,-9.382462,-5.262269,-11.371385,35.828892


## 中文结论

In [3]:
display(Markdown((CARD / 'conclusion.md').read_text(encoding='utf-8')))

# 中文结论

- 做了什么：把原模型的 SOXL/SOXX/cash 合法仓位、alpha/tail 概率、组合自身回撤状态与开盘后前四根 15 分钟路径合并；模型在 10:30 直接重新输出三资产仓位，不再使用底仓加 overlay。
- 为什么之前现金过多：日频回撤状态机只能按整天降风险，错过反弹后又重复进入防守；收益损失来自现金时点，而不只是平均现金比例。
- 训练结构：教师标签判断“恢复到 1.2 beta 是否有正增量”，实际执行只恢复到 0.6 beta，用不对称 teacher-student 校准限制误判杠杆。
- 选择规则：只使用 2019-2022 开发期年度/两年窗口；2023-2026 outer 完全不参与模型、阈值或动作 beta 选择。
- 外层 10bp：return=629.6689%，MaxDD=-28.1051%，worst day=-9.3825%，CVaR5=-5.2538%，worst 5d=-11.3018%。
- 现金与收益：日频控制器 outer return=540.5320%，新模型提高 89.1369 个百分点；平均现金减少 1.9309 个百分点。
- 25bp 冻结决策压力：return=557.4358%，MaxDD=-29.0887%；只重算成本，不允许成本改变仓位路径。
- 效率门：return/MaxDD=22.4041，同平均风险 SOXL=16.0796，matched-beta=10.4061，均通过。旧版 SOXL upside capture=34.5870% 仍未过 60% 门；该旧门不属于当前“超过 SOXX 且保留现金”目标。
- 原模型差距：原模型 outer return=707.5934%，新模型仍少 77.9245 个百分点；尚未做到完全保持原收益。
- 是否晋升：promoted_research_candidate_intraday_recovery_direct_allocation。这是研究候选晋级，不是生产或实盘成交证明；仍缺 bid/ask、排队、延迟和 10:30 真实成交 replay。
- qlib recorder：e5c1bf5e491947938535fca3ec490949
- 不构成买卖建议。
